In [19]:
# 셀 0 — 설정 (여기만 수정). 처음엔 EPOCHS=1 로 끝까지 도는지 확인하세요.
VARIANT   = "text_swap"      # "text_swap" | "none"(대조군: in-batch negative만)
LOSS      = "sigmoid"        # "sigmoid"(SigLIP 기본) | "infonce"
MODEL_ID  = "google/siglip2-large-patch16-384"

# --- LoRA ---
LORA_R, LORA_ALPHA = 16, 32
LORA_DROPOUT = 0.05          # [확신 없음] 지정 없던 값. 0.0으로 바꿔도 됨
# 텍스트/비전 encoder의 self_attn q/k/v/out_proj 에만 적용하는 정규식.
# (비전 pooling head의 nn.MultiheadAttention.out_proj 는 LoRA가 무시/오작동할 수 있어 일부러 제외)
# 리스트(["q_proj","k_proj","v_proj","out_proj"])로 바꿔도 되지만 위 head까지 걸릴 수 있음.
LORA_TARGET_MODULES = r".*(text_model|vision_model)\.encoder\.layers\.\d+\.self_attn\.(q_proj|k_proj|v_proj|out_proj)"

# --- 학습 ---
LR            = 1e-4         # v2: 2e-5는 LoRA에 낮아 효과가 없었음 → 1e-4 (이후 불안정하면 5e-5)
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.05
EPOCHS        = 3            # ← 첫 실행은 1로
BATCH_SIZE    = 32           # 1회 forward 배치. OOM이면 16으로 낮추고 GRAD_ACCUM=2
GRAD_ACCUM    = 1            # 주의: 누적하면 in-batch negative 수는 BATCH_SIZE 기준으로 줄어듦
K_NEG         = 2            # 이미지당 text_swap negative 개수
SEED          = 42
GRAD_CLIP     = 1.0
NUM_WORKERS   = 4
MAX_LEN       = 64           # SigLIP2 텍스트 max position = 64 (늘릴 수 없음)
LOWERCASE     = True         # [확신 없음] SigLIP2는 소문자 텍스트로 학습됐다고 알고 있음. 추론 서버도 동일해야 함!
MAX_STEPS_PER_EPOCH = None   # 디버그용(예: 5). 정상 실행은 None

# --- v2 추가 ---
HARD_BATCH   = True    # train 이미지를 zero-shot 임베딩으로 클러스터링 → 비슷한 이미지끼리 한 배치로 (hard in-batch negative)
CLUSTER_SIZE = 64      # 클러스터당 평균 이미지 수. 작을수록 배치 안 이미지가 더 비슷해짐(너무 작으면 가짜 negative↑)
N_TRAIN_EVAL = 300     # 훈련셋 캡션교체정확도 측정용 샘플 수(과적합/학습신호 진단용)
N_BOOT       = 1000    # val 지표 부트스트랩 신뢰구간 반복 수
FALLBACK_TO_BASE = True  # 어떤 epoch도 zero-shot보다 못하면 원본(zero-shot) 모델을 fp16으로 저장

# --- 어휘 추출 ---
MIN_WORD_COUNT = 5           # 색/숫자 단어를 목록에 넣을 최소 등장 횟수 (train 캡션 기준)
MIN_OBJ_COUNT, TOP_OBJECTS = 20, 60

# --- 경로/기타 ---
RESUME    = True
WORK      = "/kaggle/working"
CKPT_DIR  = f"{WORK}/ckpt_{VARIANT}"
# 최종 제출용 폴더. text_swap이 아닌 variant는 덮어쓰기 방지를 위해 접미사를 붙임.
FINAL_DIR = f"{WORK}/finetuned_model" if VARIANT == "text_swap" else f"{WORK}/finetuned_model_{VARIANT}"
LOG_EVERY = 10

In [20]:
# 셀 1 — 환경/임포트/시드
import os, sys, json, math, time, random, re, glob, gc, difflib, collections, subprocess
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np, pandas as pd, torch, torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

try:
    import peft
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "peft"])
    import peft
import transformers
from transformers import AutoModel, AutoProcessor
from peft import LoraConfig, get_peft_model, PeftModel

# Kaggle 이미지의 torchao(0.10)가 peft가 요구하는 버전(>=0.16)보다 낮아 get_peft_model에서 ImportError가 남.
# torchao는 이 노트북에서 쓰지 않으므로 peft의 torchao 감지를 꺼서 우회 (커널 재시작 불필요).
try:
    import importlib.metadata as _md
    from packaging.version import Version as _V
    _tv = _md.version("torchao")
    if _V(_tv) < _V("0.16.0"):
        import peft.tuners.lora.torchao  # noqa  (모듈 로드해서 함께 패치)
        for _m in list(sys.modules.values()):
            if _m is not None and getattr(_m, "__name__", "").startswith("peft") and hasattr(_m, "is_torchao_available"):
                _m.is_torchao_available = lambda *a, **k: False
        print(f"torchao {_tv} 감지 → peft torchao 경로 비활성화(패치)")
except Exception as _e:
    pass
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
# [확신 없음] SigLIP2 체크포인트 로딩에 transformers>=4.49 권장. 로딩 에러가 나면
#   !pip install -U transformers  후 커널 재시작.
assert torch.cuda.is_available(), "GPU가 꺼져 있습니다 (Accelerator → GPU T4)"
device = "cuda"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))

torchao 0.10.0 감지 → peft torchao 경로 비활성화(패치)
torch 2.11.0+cu130 | transformers 5.20.2 | peft 0.21.2
GPU: Tesla T4


In [21]:
# 셀 2 — 데이터 로드, 컬럼 자동 탐지, validation 분리
CSVS = glob.glob("/kaggle/input/**/prompts.csv", recursive=True)
assert CSVS, "prompts.csv 를 /kaggle/input 아래에서 못 찾음 (데이터셋 추가 확인)"
CSV_PATH = sorted(CSVS, key=lambda p: ("haikuthon" not in p.lower(), len(p)))[0]
DATA_ROOT = os.path.dirname(CSV_PATH)
print("CSV:", CSV_PATH)

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
def _num_key(s):                              # "img_0012.jpg" / "0012" / 12 → 12 (마지막 숫자 덩어리)
    m = re.findall(r"\d+", s)
    return int(m[-1]) if m else None

def build_index(root):
    full, name, stem, num = {}, {}, {}, {}
    dup = set()
    for dp, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(IMG_EXT):
                p = os.path.join(dp, f)
                full[os.path.relpath(p, root)] = p
                name.setdefault(f, p); st = os.path.splitext(f)[0]; stem.setdefault(st, p)
                k = _num_key(st)
                if k is not None:
                    if k in num: dup.add(k)
                    num[k] = p
    for k in dup: num.pop(k, None)             # 숫자 키가 겹치면 모호하므로 제외
    return full, name, stem, num
IDX_FULL, IDX_NAME, IDX_STEM, IDX_NUM = build_index(DATA_ROOT)
if len(IDX_NAME) < 10:                       # images/ 가 csv와 다른 위치일 경우 상위 폴더에서 재탐색
    IDX_FULL, IDX_NAME, IDX_STEM, IDX_NUM = build_index(os.path.dirname(DATA_ROOT))
print("발견한 이미지 파일 수:", len(IDX_NAME), "| 예시:", list(IDX_NAME)[:3], "| DATA_ROOT:", DATA_ROOT)

def resolve(v):
    s = str(v).strip()
    if s in IDX_FULL: return IDX_FULL[s]
    b = os.path.basename(s)
    r = IDX_NAME.get(b) or IDX_STEM.get(os.path.splitext(b)[0])
    if r: return r
    if len(s) <= 40 and " " not in s:         # id 컬럼처럼 짧은 값만 숫자 매칭 (긴 캡션 오매칭 방지)
        try:
            k = _num_key(s)
            if k is not None: return IDX_NUM.get(k)
        except Exception: pass
    return None

df = pd.read_csv(CSV_PATH)                   # ※ 행 순서/개수 절대 변경 금지 (팀원과 동일 분할)
print("df shape:", df.shape, "| columns:", list(df.columns))

# 이미지 컬럼: 값이 실제 이미지 파일로 연결되는 컬럼 (확장자가 있는 컬럼을 우선)
cands = []
for c in df.columns:
    vals = df[c].dropna().head(300).tolist()
    if not vals: continue
    hit = np.mean([resolve(v) is not None for v in vals])
    ext = np.mean([os.path.splitext(str(v))[1].lower() in IMG_EXT for v in vals])
    cands.append((hit + 0.5 * ext, hit, c))
    print(f"  col={c!r:20} 이미지 연결률={hit:.2f} 확장자포함={ext:.2f}")
cands.sort(reverse=True)
print("id 예시:", df.iloc[:3, 0].tolist())
assert cands and cands[0][1] >= 0.8, ("이미지 컬럼 자동 탐지 실패 → 위의 '이미지 파일 수/예시'와 'id 예시'를 확인하세요 "
                                       "(이미지 수가 0이면 DATA_ROOT/데이터셋 추가 문제)")
IMG_COL = cands[0][2]
# 캡션 컬럼: 이미지 컬럼 제외, 문자열 컬럼 중 평균 길이가 가장 긴 것
obj_cols = [c for c in df.columns if c != IMG_COL and not pd.api.types.is_numeric_dtype(df[c])]
CAP_COL = max(obj_cols, key=lambda c: df[c].fillna("").astype(str).str.len().mean())
print(f"→ 이미지 컬럼={IMG_COL!r}, 캡션 컬럼={CAP_COL!r}")

all_paths = [resolve(v) for v in df[IMG_COL]]
missing = [i for i, p in enumerate(all_paths) if p is None]
assert not missing, f"이미지 파일을 못 찾은 행 {len(missing)}개 (예: {missing[:5]})"
all_caps = df[CAP_COL].fillna("").astype(str).tolist()
if len(df) != 4000: print("⚠️ 행 수가 4000이 아닙니다:", len(df))

# ===== validation 분리 (요구사항 그대로) =====
idx = np.random.RandomState(42).permutation(len(df))
val_idx = sorted(idx[:500].tolist())
train_idx = sorted(idx[500:].tolist())
assert not (set(val_idx) & set(train_idx))
json.dump(val_idx, open(f"{WORK}/val_idx.json", "w"))
print("val_idx[:10] =", val_idx[:10], " ← 팀원과 동일한지 확인")
print("train:", len(train_idx), "| val:", len(val_idx))

train_paths = [all_paths[i] for i in train_idx]; train_caps = [all_caps[i] for i in train_idx]
val_paths   = [all_paths[i] for i in val_idx];   val_caps   = [all_caps[i] for i in val_idx]
print("캡션 예시:", train_caps[0][:300])

CSV: /kaggle/input/datasets/dahyvvn/haikuthon/prompts.csv
발견한 이미지 파일 수: 4000 | 예시: ['0000.png', '0001.png', '0002.png'] | DATA_ROOT: /kaggle/input/datasets/dahyvvn/haikuthon
df shape: (4000, 2) | columns: ['id', 'prompt']
  col='id'                 이미지 연결률=1.00 확장자포함=0.00
  col='prompt'             이미지 연결률=0.00 확장자포함=0.00
id 예시: [0, 1, 2]
→ 이미지 컬럼='id', 캡션 컬럼='prompt'
val_idx[:10] = [8, 12, 14, 17, 23, 26, 32, 33, 44, 45]  ← 팀원과 동일한지 확인
train: 3500 | val: 500
캡션 예시: One person is lying on an airport apron. One scarf is solid-colored and gray. The person is wearing the gray scarf. One scarf is polka-dotted and orange. The person is above the orange scarf. The scene is a photograph from a front view with side lighting.


In [22]:
# 셀 3 — 어휘 추출(train 캡션만) + text_swap 생성기
TOKEN_RE = re.compile(r"[A-Za-z]+|\d+")
COLOR_LEXICON = ["red","orange","yellow","green","blue","purple","pink","brown","black","white",
                 "gray","grey","silver","gold","golden","beige","cyan","teal","turquoise","violet",
                 "maroon","navy","cream","tan","magenta","lavender","olive","ivory","crimson",
                 "scarlet","indigo","coral","amber","bronze","copper","aqua","burgundy","khaki","peach","mint"]
# 거의 같은 색(동의어)끼리 바꾸면 "정답과 같은 뜻"의 가짜 negative가 되므로 같은 그룹으로 묶어 제외
COLOR_SYN_GROUPS = [{"gray","grey"},{"gold","golden"},{"purple","violet"},
                    {"cyan","aqua","turquoise"},{"beige","tan","cream","ivory"}]
NUM_WORDS = {"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,"ten":10,"eleven":11,"twelve":12}
# ※ "one"/"1"은 일부러 제외: "one cat"→"two cat" 처럼 복수형 문법이 깨지기 때문.

word_cnt  = collections.Counter(w.lower() for c in train_caps for w in TOKEN_RE.findall(c) if w.isalpha())
digit_cnt = collections.Counter(w for c in train_caps for w in TOKEN_RE.findall(c) if w.isdigit())

colors      = [w for w in COLOR_LEXICON if word_cnt[w] >= MIN_WORD_COUNT]
num_words   = [w for w in NUM_WORDS if word_cnt[w] >= MIN_WORD_COUNT]
num_digits  = [d for d in map(str, range(2, 13)) if digit_cnt[d] >= MIN_WORD_COUNT]
try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as _STOP
    STOP = set(_STOP)
except Exception:
    STOP = set("a an the and of in on at with to is are was were be been it its this that these those by for from as or "
               "has have had there their they he she his her them which who whom while into onto over under near next "
               "also very more most some any each other another both one two".split())
# [확신 없음] 품사 태깅 없이 빈도 기반이라 목록에 형용사/동사가 섞일 수 있음. 아래 text_swap에서는 쓰지 않고 확인용으로만 저장.
object_words = [w for w, c in word_cnt.most_common()
                if len(w) >= 3 and w not in STOP and w not in COLOR_LEXICON and w not in NUM_WORDS
                and c >= MIN_OBJ_COUNT][:TOP_OBJECTS]
print("색 단어   :", colors)
print("숫자 단어 :", num_words, "| 숫자 표기:", num_digits)
print("물체 단어(상위):", object_words)
json.dump(dict(colors=colors, num_words=num_words, num_digits=num_digits, objects=object_words),
          open(f"{WORK}/vocab_train.json", "w"), ensure_ascii=False, indent=1)


class Swapper:
    """캡션의 색/숫자 단어를 바꿔 hard negative 생성.
    ops: color_replace(색 하나 교체), color_swap(두 색 위치 교환), num_replace, num_swap"""
    def __init__(self, colors, num_words, num_digits, tok=None, max_len=64, lowercase=True):
        self.colors = list(colors); self.color_set = set(colors)
        gid = {w: k for k, g in enumerate(COLOR_SYN_GROUPS) for w in g}
        self.cgroup = {c: gid.get(c, 1000 + i) for i, c in enumerate(self.colors)}
        self.num_words, self.num_digits = list(num_words), list(num_digits)
        self.num_word_set, self.num_digit_set = set(num_words), set(num_digits)
        self.tok, self.max_len, self.lowercase = tok, max_len, lowercase

    @staticmethod
    def _case(orig, new):
        if len(orig) > 1 and orig.isupper(): return new.upper()
        if orig[:1].isupper(): return new.capitalize()
        return new

    @staticmethod
    def _nval(w):
        lw = w.lower()
        return NUM_WORDS[lw] if lw in NUM_WORDS else int(w)

    def _scan(self, text):
        toks = [(m.start(), m.end(), m.group()) for m in TOKEN_RE.finditer(text)]
        col, num = [], []
        for i, (s, e, w) in enumerate(toks):
            lw = w.lower()
            if lw in self.color_set:
                col.append(i)
            elif lw in self.num_word_set:
                num.append(i)
            elif w.isdigit() and w in self.num_digit_set:
                before = text[s-1] if s > 0 else " "; after = text[e] if e < len(text) else " "
                if not (before.isalnum() or before in "-.") and not (after.isalnum() or after == "-"):
                    num.append(i)       # "2D", "3-year-old" 같은 경우 제외
        return toks, col, num

    def _pool(self, w):
        return self.num_words if w.lower() in NUM_WORDS else self.num_digits

    def _render(self, text, toks, repl):
        out, pos = [], 0
        for i, (s, e, w) in enumerate(toks):
            if i in repl:
                out.append(text[pos:s]); out.append(repl[i]); pos = e
        out.append(text[pos:])
        return "".join(out)

    def _fix_articles(self, text, toks, repl, replaced):
        for i in replaced:                       # a orange → an orange / an blue → a blue
            if i == 0 or (i - 1) in repl: continue
            ps, pe, pw = toks[i-1]
            gap = text[pe:toks[i][0]]
            if pw.lower() in ("a", "an") and gap and gap.isspace():
                want = "an" if repl[i][0].lower() in "aeiou" else "a"
                if want != pw.lower(): repl[i-1] = self._case(pw, want)

    def _valid(self, text, neg):
        if neg == text or (self.lowercase and neg.lower() == text.lower()): return False
        if self.tok is not None:                 # 64토큰 truncation 뒤에도 달라야 함 (뒤쪽만 바뀐 negative 방지)
            f = (lambda s: s.lower()) if self.lowercase else (lambda s: s)
            a = self.tok(f(text), max_length=self.max_len, truncation=True)["input_ids"]
            b = self.tok(f(neg),  max_length=self.max_len, truncation=True)["input_ids"]
            if a == b: return False
        return True

    def make(self, text, rng):
        toks, col, num = self._scan(text)
        ops = []
        if col: ops.append("color_replace")
        if len({self.cgroup[toks[i][2].lower()] for i in col}) >= 2: ops.append("color_swap")
        if any(len(self._pool(toks[i][2])) >= 2 for i in num): ops.append("num_replace")
        if len({self._nval(toks[i][2]) for i in num}) >= 2: ops.append("num_swap")
        rng.shuffle(ops)
        for op in ops:
            repl, replaced = {}, []
            if op == "color_replace":
                i = rng.choice(col); cur = self.cgroup[toks[i][2].lower()]
                cands = [c for c in self.colors if self.cgroup[c] != cur]
                repl[i] = self._case(toks[i][2], rng.choice(cands)); replaced = [i]
            elif op == "color_swap":
                pairs = [(a, b) for a in col for b in col if a < b and
                         self.cgroup[toks[a][2].lower()] != self.cgroup[toks[b][2].lower()]]
                a, b = rng.choice(pairs)
                repl[a] = self._case(toks[a][2], toks[b][2].lower())
                repl[b] = self._case(toks[b][2], toks[a][2].lower()); replaced = [a, b]
            elif op == "num_replace":
                ok = [i for i in num if len(self._pool(toks[i][2])) >= 2]
                i = rng.choice(ok); cur = self._nval(toks[i][2])
                cands = [x for x in self._pool(toks[i][2]) if self._nval(x) != cur]
                repl[i] = self._case(toks[i][2], rng.choice(cands))
            else:  # num_swap
                pairs = [(a, b) for a in num for b in num if a < b and
                         self._nval(toks[a][2]) != self._nval(toks[b][2])]
                a, b = rng.choice(pairs)
                repl[a] = self._case(toks[a][2], toks[b][2].lower()); repl[b] = self._case(toks[b][2], toks[a][2].lower())
            if replaced: self._fix_articles(text, toks, repl, replaced)
            neg = self._render(text, toks, repl)
            if self._valid(text, neg): return neg
        return None

swapper = Swapper(colors, num_words, num_digits, tok=None, max_len=MAX_LEN, lowercase=LOWERCASE)

색 단어   : ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink', 'brown', 'black', 'white', 'gray', 'beige']
숫자 단어 : ['two', 'three'] | 숫자 표기: ['3']
물체 단어(상위): ['lighting', 'scene', 'view', 'studio', 'plain', 'backdrop', 'person', 'overhead', 'viewed', 'soft', 'diffuse', 'painting', 'indoor', 'shown', 'warm', 'render', 'direct', 'sunlight', 'right', 'shows', 'left', 'oblique', 'photograph', 'colored', 'pencil', 'open', 'present', 'drawing', 'standing', 'rests', 'rendered', 'flat', 'illustration', 'watercolor', 'inside', 'solid', 'striped', 'setting', 'polka', 'dotted', 'checkered', 'holds', 'oil', 'room', 'gouache', 'lying', 'sitting', 'farm', 'metal', 'single', 'cabinet', 'park', 'box', 'set', 'bench', 'bowl', 'low', 'poly', 'desk', 'paper']


In [23]:
# 셀 4 — processor/tokenizer, 토큰 길이 점검, negative 예시 10개 출력
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer
swapper.tok = tok

enc = tok([c.lower() if LOWERCASE else c for c in train_caps], add_special_tokens=True)["input_ids"]
L = np.array([len(x) for x in enc])
print(f"캡션 토큰 길이: mean={L.mean():.0f} p50={np.percentile(L,50):.0f} p90={np.percentile(L,90):.0f} max={L.max()} "
      f"| {MAX_LEN} 초과(잘림) 비율={np.mean(L > MAX_LEN)*100:.1f}%")
if np.mean(L > MAX_LEN) > 0.3:
    print("⚠️ 캡션 상당수가 잘립니다. 64토큰 뒤의 속성은 모델이 볼 수 없고, 해당 위치의 swap은 자동으로 제외됩니다.")

def show_diff(a, b):
    ta, tb = a.split(), b.split()
    sm = difflib.SequenceMatcher(None, ta, tb)
    return ", ".join(f"{' '.join(ta[i1:i2])}→{' '.join(tb[j1:j2])}" for t, i1, i2, j1, j2 in sm.get_opcodes() if t != "equal")

rng_demo = random.Random(SEED)
order = list(range(len(train_caps))); rng_demo.shuffle(order)
shown, n_ok = 0, 0
for i in order:
    neg = swapper.make(train_caps[i], random.Random(i))
    if neg is None: continue
    if shown < 10:
        shown += 1
        print(f"\n[{shown}] 원래: {train_caps[i]}\n    → 바꾼: {neg}\n    (변경: {show_diff(train_caps[i], neg)})")
cover = np.mean([swapper.make(c, random.Random(k)) is not None for k, c in enumerate(train_caps)])
print(f"\ntrain 캡션 중 negative 생성 가능 비율: {cover*100:.1f}%")

캡션 토큰 길이: mean=44 p50=43 p90=55 max=81 | 64 초과(잘림) 비율=2.1%

[1] 원래: The scene is rendered as a paper-cut collage from an overhead view in direct sunlight. In a park are one blue striped single sock, which is next to a checkered shirt, one other checkered shirt, and one standing red bird.
    → 바꾼: The scene is rendered as a paper-cut collage from an overhead view in direct sunlight. In a park are one red striped single sock, which is next to a checkered shirt, one other checkered shirt, and one standing blue bird.
    (변경: blue→red, red→blue)

[2] 원래: Above one gray turtle, one yellow parrot rests on one red shelf, which supports the parrot. The scene is in a studio with one plain backdrop. It is a clay-style 3D render shown from a side view under warm indoor lighting.
    → 바꾼: Above one yellow turtle, one gray parrot rests on one red shelf, which supports the parrot. The scene is in a studio with one plain backdrop. It is a clay-style 3D render shown from a side view under warm indoo

In [24]:
# 셀 5 — 데이터셋/로더, 모델+LoRA, 평가 함수, zero-shot
def load_pixels(path):
    with Image.open(path) as im:
        im = im.convert("RGB")
    return processor.image_processor(im, return_tensors="pt")["pixel_values"][0]   # SigLIP2 processor와 동일 전처리

def tokenize(texts):
    texts = [t.lower() for t in texts] if LOWERCASE else list(texts)
    # SigLIP 계열은 padding="max_length"(64)로 학습됨. attention_mask는 쓰지 않음(원 구현과 동일).
    return tok(texts, padding="max_length", max_length=MAX_LEN, truncation=True, return_tensors="pt")["input_ids"]

class TrainDS(Dataset):
    def __init__(self, paths, caps, variant, k, seed):
        self.paths, self.caps, self.variant, self.k, self.seed, self.epoch = paths, caps, variant, k, seed, 0
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        cap, negs = self.caps[i], []
        if self.variant == "text_swap":          # 에폭마다 다른 negative (seed+epoch+idx 로 재현 가능)
            rng = random.Random(self.seed * 1_000_003 + self.epoch * 10_007 + i)
            seen = {cap}
            for _ in range(self.k * 6):
                if len(negs) >= self.k: break
                n = swapper.make(cap, rng)
                if n is not None and n not in seen: seen.add(n); negs.append(n)
        return load_pixels(self.paths[i]), cap, negs

def train_collate(batch):
    pix = torch.stack([b[0] for b in batch])
    pos = [b[1] for b in batch]; negs = [n for b in batch for n in b[2]]
    return pix, tokenize(pos + negs), len(pos)          # 앞 B개 = 정답 캡션, 뒤 = 추가 negative

class ImgDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return load_pixels(self.paths[i])

train_ds = TrainDS(train_paths, train_caps, VARIANT, K_NEG, SEED)

# ---- 모델 + LoRA ----
base = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(device)
# fp32로 로드: GradScaler는 fp16 파라미터의 grad를 unscale 못 함. autocast로 fp16 연산. (3.5GB 정도)
for p in base.parameters(): p.requires_grad = False        # 원래 가중치 고정 (logit_scale/bias 포함)
try:
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except Exception as e:
    print("⚠️ gradient checkpointing 활성화 실패(확신 없음):", e)
pm = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                     target_modules=LORA_TARGET_MODULES, bias="none"))
pm.print_trainable_parameters()
n_lora = sum(1 for n, _ in pm.named_parameters() if "lora_A" in n)
print("LoRA 적용 모듈 수:", n_lora, "(large: 24층×4×2타워=192 예상, [확신 없음])")
assert n_lora > 0, "LoRA가 한 곳에도 안 붙음 → LORA_TARGET_MODULES 확인"

def _t(x):  # transformers 버전에 따라 텐서 또는 output 객체가 올 수 있어 방어 [확신 없음]
    return x if torch.is_tensor(x) else x.pooler_output
def img_feat(m, pix): return _t(m.get_image_features(pixel_values=pix))
def txt_feat(m, ids): return _t(m.get_text_features(input_ids=ids))
AC = lambda: torch.autocast("cuda", dtype=torch.float16)

# ---- 평가 ----
@torch.no_grad()
def embed_images(m, paths):
    out = []
    for pix in DataLoader(ImgDS(paths), batch_size=32, shuffle=False, num_workers=NUM_WORKERS):
        with AC(): f = img_feat(m, pix.to(device))
        out.append(F.normalize(f.float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def embed_texts(m, texts):
    out = []
    for i in range(0, len(texts), 100):
        with AC(): f = txt_feat(m, tokenize(texts[i:i+100]).to(device))
        out.append(F.normalize(f.float(), dim=-1))
    return torch.cat(out)

# 캡션 교체 평가용 negative: val 캡션에 대해 고정 seed로 1개씩 (variant/epoch와 무관하게 항상 동일).
# ※ 단어 목록은 train 기준. val은 평가용 negative 생성에만 쓰며 학습에는 쓰지 않음.
val_swap = {}
for j, c in enumerate(val_caps):
    n = swapper.make(c, random.Random(10_000 + j))
    if n is not None: val_swap[j] = n
val_swap_ids = sorted(val_swap)
print(f"캡션 교체 평가 가능한 val 샘플: {len(val_swap_ids)}/{len(val_caps)}")

def _boot_ci(x, n=N_BOOT, seed=0):
    x = np.asarray(x, dtype=np.float64)
    if len(x) == 0: return [float("nan"), float("nan")]
    r = np.random.default_rng(seed).integers(0, len(x), (n, len(x)))
    m = x[r].mean(1)
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

# [v3] 어려운 갤러리 평가: val 이미지를 zero-shot 임베딩으로 10그룹(비슷한 이미지끼리)으로 묶고,
# 각 질의를 "같은 그룹 이미지(약 50장)"끼리만 비교해 순위를 계산. 랜덤 500장보다 훨씬 변별력이 큼.
# (val은 평가에만 사용. 그룹은 원본 모델 기준이라 variant/epoch와 무관하게 고정)
from sklearn.cluster import KMeans
_val_img0 = embed_images(pm, val_paths).cpu().numpy()          # 이 시점의 pm = 원본(LoRA B=0)
VAL_GROUP = torch.tensor(KMeans(n_clusters=10, n_init=3, random_state=0).fit(_val_img0).labels_, device=device)
print("val 그룹 크기:", torch.bincount(VAL_GROUP).tolist())

def _metrics(rank):
    r1 = (rank == 1).float(); nd = (rank <= 5).float() / torch.log2(rank + 1)
    return r1.cpu().numpy(), nd.cpu().numpy()

@torch.no_grad()
def evaluate(m):
    m.eval()
    img = embed_images(m, val_paths); txt = embed_texts(m, val_caps)
    S = txt @ img.T                                           # [텍스트500, 이미지500]
    beat = S > S.diag().unsqueeze(1)
    r1v, ndv = _metrics(1 + beat.sum(1).float()); fin = 0.5 * (r1v + ndv)
    same = VAL_GROUP[:, None] == VAL_GROUP[None, :]
    hr1, hnd = _metrics(1 + (beat & same).sum(1).float()); hard = 0.5 * (hr1 + hnd)
    res = dict(recall1=float(r1v.mean()), ndcg5=float(ndv.mean()), final=float(fin.mean()), ci_final=_boot_ci(fin),
               hard_final=float(hard.mean()), ci_hard=_boot_ci(hard), fin_vec=fin.tolist(), hard_vec=hard.tolist(),
               swap_acc=float("nan"), ci_swap=[float("nan")] * 2, swap_n=len(val_swap_ids))
    if val_swap_ids:
        ids = torch.tensor(val_swap_ids, device=img.device)
        sw = embed_texts(m, [val_swap[j] for j in val_swap_ids])
        true_sim = (txt[ids] * img[ids]).sum(-1); sw_sim = (sw * img[ids]).sum(-1)
        accv = (true_sim > sw_sim).float().cpu().numpy()
        res["swap_acc"] = float(accv.mean()); res["ci_swap"] = _boot_ci(accv)
    return res

def delta_str(res, zs):
    """zero-shot 대비 개선폭 (같은 질의끼리 짝지은 paired 부트스트랩 95% CI). 구간이 0을 포함하면 개선/악화라고 단정 불가."""
    out = []
    for name, k in (("FINAL", "fin_vec"), ("hard-FINAL", "hard_vec")):
        d = np.array(res[k]) - np.array(zs[k]); lo, hi = _boot_ci(d)
        out.append(f"Δ{name}={d.mean():+.4f} [{lo:+.4f}~{hi:+.4f}]")
    return " | ".join(out)

# 훈련셋 일부로 같은 캡션교체 정확도 측정: 훈련 쪽만 오르면 과적합, 둘 다 안 오르면 학습 신호 문제
_sub = sorted(random.Random(SEED).sample(range(len(train_paths)), min(N_TRAIN_EVAL, len(train_paths))))
TR_SWAP = {}
for j in _sub:
    n = swapper.make(train_caps[j], random.Random(20_000 + j))
    if n is not None: TR_SWAP[j] = n
TR_IDS = sorted(TR_SWAP)

@torch.no_grad()
def train_swap_acc(m):
    m.eval()
    if not TR_IDS: return float("nan")
    img = embed_images(m, [train_paths[j] for j in TR_IDS])
    t = embed_texts(m, [train_caps[j] for j in TR_IDS]); sw = embed_texts(m, [TR_SWAP[j] for j in TR_IDS])
    return float(((t * img).sum(-1) > (sw * img).sum(-1)).float().mean())

def fmt(r):
    return (f"FINAL={r['final']:.4f} [95%CI {r['ci_final'][0]:.3f}~{r['ci_final'][1]:.3f}] | Recall@1={r['recall1']:.4f} | "
            f"NDCG@5={r['ndcg5']:.4f} | hard-FINAL={r['hard_final']:.4f} [{r['ci_hard'][0]:.3f}~{r['ci_hard'][1]:.3f}] | "
            f"캡션교체정확도={r['swap_acc']:.4f} [{r['ci_swap'][0]:.3f}~{r['ci_swap'][1]:.3f}] (n={r['swap_n']})")

# ---- zero-shot (LoRA B=0 이라 base와 동일). 이어하기 시 재계산하지 않도록 파일로 보관 ----
os.makedirs(CKPT_DIR, exist_ok=True)
ZS_PATH = f"{CKPT_DIR}/zero_shot.json"
zero_shot = json.load(open(ZS_PATH)) if (RESUME and os.path.exists(ZS_PATH)) else None
if zero_shot is None or any(k not in zero_shot for k in ("ci_final", "hard_final", "fin_vec", "train_swap_acc")):   # 구버전 캐시면 재계산
    zero_shot = evaluate(pm); zero_shot["train_swap_acc"] = train_swap_acc(pm)
    json.dump(zero_shot, open(ZS_PATH, "w"))
print("[zero-shot]", fmt(zero_shot), f"| 훈련셋 캡션교체={zero_shot['train_swap_acc']:.4f}")

# ---- hard batch용 train 이미지 zero-shot 임베딩 (LoRA 로드 전, 즉 원본 상태에서 1회 계산 후 캐시) ----
TRAIN_EMB_PATH = f"{CKPT_DIR}/train_emb_zeroshot.pt"
train_emb = None
if HARD_BATCH:
    if os.path.exists(TRAIN_EMB_PATH):
        train_emb = torch.load(TRAIN_EMB_PATH, map_location="cpu")
    if train_emb is None or len(train_emb) != len(train_paths):
        train_emb = embed_images(pm, train_paths).cpu(); torch.save(train_emb, TRAIN_EMB_PATH)
    print("train 임베딩:", tuple(train_emb.shape))

Loading weights:   0%|          | 0/792 [00:00<?, ?it/s]

trainable params: 6,291,456 || all params: 888,145,922 || trainable%: 0.7084
LoRA 적용 모듈 수: 192 (large: 24층×4×2타워=192 예상, [확신 없음])
캡션 교체 평가 가능한 val 샘플: 468/500
val 그룹 크기: [30, 35, 52, 86, 51, 44, 30, 55, 43, 74]
[zero-shot] FINAL=0.8493 [95%CI 0.824~0.874] | Recall@1=0.7860 | NDCG@5=0.9126 | hard-FINAL=0.8697 [0.845~0.893] | 캡션교체정확도=0.7521 [0.712~0.793] (n=468) | 훈련셋 캡션교체=0.7687
train 임베딩: (3500, 1024)


In [25]:
# 셀 6 — 학습 (epoch마다 평가/체크포인트, 이어하기 지원)
CFG = dict(VARIANT=VARIANT, LOSS=LOSS, MODEL_ID=MODEL_ID, LORA_R=LORA_R, LORA_ALPHA=LORA_ALPHA,
           LORA_DROPOUT=LORA_DROPOUT, LORA_TARGET_MODULES=LORA_TARGET_MODULES, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY,
           WARMUP_RATIO=WARMUP_RATIO, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM, K_NEG=K_NEG,
           SEED=SEED, MAX_LEN=MAX_LEN, LOWERCASE=LOWERCASE, MAX_STEPS_PER_EPOCH=MAX_STEPS_PER_EPOCH,
           HARD_BATCH=HARD_BATCH, CLUSTER_SIZE=CLUSTER_SIZE)

from sklearn.cluster import KMeans
class ClusterBatchSampler(torch.utils.data.Sampler):
    """에폭마다 (seed+epoch)로 KMeans를 다시 돌려, 같은 클러스터(=비슷한 이미지)가 한 배치에 모이도록 배치를 구성."""
    def __init__(self, emb, batch_size, cluster_size, seed):
        self.emb = emb.numpy(); self.bs = batch_size; self.seed = seed; self.epoch = 0
        self.k = max(2, len(self.emb) // cluster_size)
    def set_epoch(self, e): self.epoch = e
    def _batches(self):
        sd = self.seed + self.epoch; rng = np.random.RandomState(sd)
        lab = KMeans(n_clusters=self.k, n_init=1, random_state=sd).fit(self.emb).labels_
        groups = [rng.permutation(np.where(lab == c)[0]) for c in range(self.k)]
        stream = np.concatenate([groups[c] for c in rng.permutation(self.k)])
        nb = len(stream) // self.bs
        batches = [stream[i * self.bs:(i + 1) * self.bs].tolist() for i in range(nb)]
        random.Random(sd).shuffle(batches)
        return batches
    def __iter__(self): return iter(self._batches())
    def __len__(self): return len(self.emb) // self.bs

sampler = ClusterBatchSampler(train_emb, BATCH_SIZE, CLUSTER_SIZE, SEED) if HARD_BATCH else None

def contrastive_loss(fi, ft, B):
    """fi:[B,D] 이미지, ft:[B+M,D] 텍스트(앞 B개=정답, 뒤 M개=추가 negative). 모두 L2 정규화."""
    logits = fi @ ft.T
    if LOSS == "sigmoid":                                      # SigLIP sigmoid loss (/B 정규화, 원 논문과 동일)
        logits = logits * base.logit_scale.exp().float() + base.logit_bias.float()
        labels = -torch.ones_like(logits); labels[torch.arange(B), torch.arange(B)] = 1.0
        return -F.logsigmoid(labels * logits).sum() / B       # negative 텍스트는 모든 이미지에 대해 label=-1
    logits = logits * base.logit_scale.exp().float()           # InfoNCE: negative는 이미지→텍스트 후보에만 추가
    t = torch.arange(B, device=logits.device)
    return 0.5 * (F.cross_entropy(logits, t) + F.cross_entropy(logits[:, :B].T, t))

trainable = [p for p in pm.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda")
steps_per_epoch = len(train_ds) // BATCH_SIZE // GRAD_ACCUM
if MAX_STEPS_PER_EPOCH: steps_per_epoch = min(steps_per_epoch, MAX_STEPS_PER_EPOCH)
total_steps = EPOCHS * steps_per_epoch; warm = max(1, int(WARMUP_RATIO * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                          else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))

state = dict(epoch_done=0, best=(-1.0, -1.0), best_epoch=None, history=[], train_time=0.0, cfg=CFG)
STATE_PATH = f"{CKPT_DIR}/state.pt"
if RESUME and os.path.exists(STATE_PATH):
    ck = torch.load(STATE_PATH, map_location="cpu", weights_only=False)
    if ck["cfg"] == CFG:
        miss, unexp = pm.load_state_dict(ck["lora"], strict=False)
        assert not unexp, f"예상치 못한 키: {unexp[:3]}"
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        state = {k: ck[k] for k in state}; state["cfg"] = CFG
        print(f"▶ 이어하기: epoch {state['epoch_done']}/{EPOCHS} 까지 완료된 체크포인트 로드")
    else:
        print("⚠️ 저장된 체크포인트와 설정(CFG)이 달라 무시하고 처음부터 시작합니다 (예: EPOCHS 변경).")

def save_state():
    lora = {n: p.detach().cpu() for n, p in pm.named_parameters() if p.requires_grad}
    pm.save_pretrained(f"{CKPT_DIR}/last_adapter")
    torch.save(dict(lora=lora, opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(), **state),
               STATE_PATH + ".tmp")
    os.replace(STATE_PATH + ".tmp", STATE_PATH)

prev = state["history"][-1] if state["history"] else zero_shot
for epoch in range(state["epoch_done"], EPOCHS):
    t0 = time.time(); train_ds.epoch = epoch
    if HARD_BATCH:
        sampler.set_epoch(epoch)
        loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=NUM_WORKERS,
                            collate_fn=train_collate, pin_memory=True)
    else:
        g = torch.Generator(); g.manual_seed(SEED + epoch)
        loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g, num_workers=NUM_WORKERS,
                            drop_last=True, collate_fn=train_collate, pin_memory=True)
    pm.train(); opt.zero_grad(set_to_none=True)
    micro = step = 0; run, nrun, n_neg = 0.0, 0, 0
    for pix, ids, B in loader:
        pix, ids = pix.to(device, non_blocking=True), ids.to(device)
        with AC():
            fi = img_feat(base, pix); ft = txt_feat(base, ids)
        loss = contrastive_loss(F.normalize(fi.float(), dim=-1), F.normalize(ft.float(), dim=-1), B)
        if not torch.isfinite(loss): print("⚠️ loss가 inf/nan → 이 배치 건너뜀"); opt.zero_grad(set_to_none=True); continue
        scaler.scale(loss / GRAD_ACCUM).backward()
        run += loss.item(); nrun += 1; n_neg += ids.shape[0] - B; micro += 1
        if micro % GRAD_ACCUM == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            if step % LOG_EVERY == 0:
                print(f"  ep{epoch+1} step {step}/{steps_per_epoch} loss={run/nrun:.4f} lr={sched.get_last_lr()[0]:.2e} "
                      f"mem={torch.cuda.max_memory_allocated()/2**30:.1f}GB"); run, nrun = 0.0, 0
            if MAX_STEPS_PER_EPOCH and step >= MAX_STEPS_PER_EPOCH: break
    print(f"  (epoch {epoch+1}: 추가 negative 텍스트 평균 {n_neg/max(1,micro):.1f}개/배치)")

    ev = evaluate(pm)
    ev["train_swap_acc"] = train_swap_acc(pm)
    print(f"[epoch {epoch+1}/{EPOCHS}] {fmt(ev)}")
    print("  zero-shot 대비:", delta_str(ev, zero_shot))
    print(f"  훈련셋 캡션교체정확도={ev['train_swap_acc']:.4f} (zero-shot {zero_shot['train_swap_acc']:.4f}) "
          "← 훈련만 오르면 과적합, 둘 다 그대로면 학습 신호 부족")
    if ev["final"] < prev["final"] - 1e-9 or (ev["swap_acc"] < prev["swap_acc"] - 1e-9):
        print("⚠️ 과적합 경고: validation 점수가 이전보다 내려갔습니다 "
              f"(FINAL {prev['final']:.4f}→{ev['final']:.4f}, 캡션교체 {prev['swap_acc']:.4f}→{ev['swap_acc']:.4f})")
    score = (round(ev["final"], 6), round(ev["swap_acc"], 6))   # FINAL이 포화되면 캡션교체 정확도로 tie-break
    if score > tuple(state["best"]):
        state["best"], state["best_epoch"] = score, epoch + 1
        pm.save_pretrained(f"{CKPT_DIR}/best_adapter"); print(f"  ★ best 갱신 (epoch {epoch+1})")
    state["history"].append(ev); state["epoch_done"] = epoch + 1
    state["train_time"] += time.time() - t0
    save_state(); prev = ev
    torch.cuda.empty_cache()
print("학습 완료. best epoch:", state["best_epoch"])
ZS_SCORE = (round(zero_shot["final"], 6), round(zero_shot["swap_acc"], 6))
if tuple(state["best"]) < ZS_SCORE:
    print("❌ 어떤 epoch도 zero-shot보다 낫지 않습니다. (FALLBACK_TO_BASE=True면 셀 7에서 원본을 저장합니다)")
else:
    print("✅ zero-shot보다 개선된 epoch이 있습니다.")

⚠️ 저장된 체크포인트와 설정(CFG)이 달라 무시하고 처음부터 시작합니다 (예: EPOCHS 변경).


/tmp/ipykernel_50/2837468377.py:92: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1


  ep1 step 10/109 loss=95.1643 lr=6.88e-05 mem=11.8GB
  ep1 step 20/109 loss=37.0349 lr=1.00e-04 mem=11.8GB
  ep1 step 30/109 loss=8.8645 lr=9.95e-05 mem=11.8GB
  ep1 step 40/109 loss=7.6897 lr=9.85e-05 mem=11.8GB
  ep1 step 50/109 loss=5.4475 lr=9.71e-05 mem=11.8GB
  ep1 step 60/109 loss=7.5740 lr=9.51e-05 mem=11.8GB
  ep1 step 70/109 loss=5.4831 lr=9.27e-05 mem=11.8GB
  ep1 step 80/109 loss=5.3858 lr=8.99e-05 mem=11.8GB
  ep1 step 90/109 loss=3.5514 lr=8.67e-05 mem=11.8GB
  ep1 step 100/109 loss=3.2065 lr=8.31e-05 mem=11.8GB
  (epoch 1: 추가 negative 텍스트 평균 57.6개/배치)
[epoch 1/3] FINAL=0.8771 [95%CI 0.854~0.900] | Recall@1=0.8240 | NDCG@5=0.9302 | hard-FINAL=0.8910 [0.867~0.913] | 캡션교체정확도=0.7970 [0.763~0.833] (n=468)
  zero-shot 대비: ΔFINAL=+0.0278 [+0.0049~+0.0493] | Δhard-FINAL=+0.0213 [+0.0005~+0.0410]
  훈련셋 캡션교체정확도=0.7799 (zero-shot 0.7687) ← 훈련만 오르면 과적합, 둘 다 그대로면 학습 신호 부족
  ★ best 갱신 (epoch 1)
  ep2 step 10/109 loss=3.1149 lr=7.53e-05 mem=11.8GB
  ep2 step 20/109 loss=2.9501 lr=7.08

In [26]:
# 셀 7 — best LoRA merge → fp16 safetensors 저장, 크기 출력, 저장본 재검증
assert os.path.exists(f"{CKPT_DIR}/best_adapter"), "best_adapter 가 없음 (셀 6 먼저 실행)"
base2 = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float32)          # CPU에서 fp32로 merge 후 fp16 변환
ZS_SCORE = (round(zero_shot["final"], 6), round(zero_shot["swap_acc"], 6))
USE_BASE = FALLBACK_TO_BASE and tuple(state["best"]) < ZS_SCORE
if USE_BASE:
    print("⚠️ 파인튜닝이 zero-shot보다 못해서 LoRA를 merge하지 않고 원본 모델을 fp16으로 저장합니다.")
    merged = base2
else:
    merged = PeftModel.from_pretrained(base2, f"{CKPT_DIR}/best_adapter").merge_and_unload()
merged = merged.half().eval()
assert not any("lora" in k for k in merged.state_dict()), "merge 후에도 lora 키가 남아 있음"

os.makedirs(FINAL_DIR, exist_ok=True)
merged.save_pretrained(FINAL_DIR, safe_serialization=True)
processor.save_pretrained(FINAL_DIR)                    # tokenizer + image processor 설정 포함
files = [(f, os.path.getsize(os.path.join(FINAL_DIR, f))) for f in sorted(os.listdir(FINAL_DIR))]
for f, s in files: print(f"  {f:40s} {s/2**20:8.1f} MB")
size_gb = sum(s for _, s in files) / 1e9
print(f"저장 폴더: {FINAL_DIR} | 총 용량: {size_gb:.2f} GB", "✅ (<=5GB)" if size_gb <= 5 else "❌ 5GB 초과!")

# 디스크에서 다시 로드(peft 없이)해서 점수 확인 = 실제 제출물 검증
del merged, base2; gc.collect(); torch.cuda.empty_cache()
chk = AutoModel.from_pretrained(FINAL_DIR, torch_dtype=torch.float16).to(device)
final_eval = evaluate(chk)
print("[저장본 재검증]", fmt(final_eval))
print("  zero-shot 대비:", delta_str(final_eval, zero_shot))
print("※ 제출 서버에서도 입력 전처리가 같아야 합니다: padding='max_length', max_length=64, "
      f"lowercase={LOWERCASE}, attention_mask 미사용")

Loading weights:   0%|          | 0/792 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  chat_template.jinja                           0.0 MB
  config.json                                   0.0 MB
  model.safetensors                          1682.1 MB
  processor_config.json                         0.0 MB
  tokenizer.json                               32.8 MB
  tokenizer_config.json                         0.0 MB
저장 폴더: /kaggle/working/finetuned_model | 총 용량: 1.80 GB ✅ (<=5GB)


Loading weights:   0%|          | 0/792 [00:00<?, ?it/s]

[저장본 재검증] FINAL=0.9004 [95%CI 0.878~0.920] | Recall@1=0.8560 | NDCG@5=0.9448 | hard-FINAL=0.9100 [0.889~0.928] | 캡션교체정확도=0.8718 [0.842~0.900] (n=468)
  zero-shot 대비: ΔFINAL=+0.0511 [+0.0259~+0.0741] | Δhard-FINAL=+0.0403 [+0.0171~+0.0615]
※ 제출 서버에서도 입력 전처리가 같아야 합니다: padding='max_length', max_length=64, lowercase=True, attention_mask 미사용


In [27]:
# 셀 8 — 팀 공유용 요약
st = json.load(open(ZS_PATH)) if os.path.exists(ZS_PATH) else zero_shot
hist = state["history"]
mins = state["train_time"] / 60
_strip = lambda d: {k: v for k, v in d.items() if not k.endswith("_vec")}
summary = dict(VARIANT=VARIANT, saved_model="base(zero-shot)" if USE_BASE else "finetuned", hparams=dict(LORA_R=LORA_R, LORA_ALPHA=LORA_ALPHA, LR=LR, EPOCHS=EPOCHS, HARD_BATCH=HARD_BATCH,
               BATCH=f"{BATCH_SIZE}x{GRAD_ACCUM}", K_NEG=K_NEG if VARIANT == "text_swap" else 0, LOSS=LOSS, SEED=SEED),
               zero_shot=_strip(st), finetuned=_strip(final_eval), best_epoch=state["best_epoch"], train_minutes=round(mins, 1))
json.dump(summary, open(f"{WORK}/summary_{VARIANT}.json", "w"), indent=1)

print("=" * 60)
print(f"VARIANT           : {VARIANT}")
print(f"하이퍼파라미터    : LoRA r={LORA_R}/alpha={LORA_ALPHA}, lr={LR}, epochs={EPOCHS}, "
      f"batch={BATCH_SIZE}(accum {GRAD_ACCUM}), K={K_NEG if VARIANT=='text_swap' else 0}, loss={LOSS}, seed={SEED}")
print(f"zero-shot FINAL   : {st['final']:.4f}  [95%CI {st['ci_final'][0]:.3f}~{st['ci_final'][1]:.3f}]   (캡션 교체 정확도 {st['swap_acc']:.4f})")
print(f"파인튜닝 FINAL    : {final_eval['final']:.4f}  [95%CI {final_eval['ci_final'][0]:.3f}~{final_eval['ci_final'][1]:.3f}]   (best epoch {state['best_epoch']})"
      + ("   ※ 저장된 모델은 원본(zero-shot)"   ㅍ  if USE_BASE else ""))
print(f"zero-shot 대비    : {delta_str(final_eval, st)}")
print(f"hard-FINAL        : {final_eval['hard_final']:.4f}  (zero-shot {st['hard_final']:.4f})  ← 비슷한 이미지끼리 비교한 더 어려운 지표")
print(f"Recall@1          : {final_eval['recall1']:.4f}")
print(f"NDCG@5            : {final_eval['ndcg5']:.4f}")
print(f"캡션 교체 정확도  : {final_eval['swap_acc']:.4f}   (n={final_eval['swap_n']})")
print(f"학습 소요 시간    : {mins:.1f}분 (epoch별 평가 포함, 이어하기 시 누적)")
print("=" * 60)

VARIANT           : text_swap
하이퍼파라미터    : LoRA r=16/alpha=32, lr=0.0001, epochs=3, batch=32(accum 1), K=2, loss=sigmoid, seed=42
zero-shot FINAL   : 0.8493  [95%CI 0.824~0.874]   (캡션 교체 정확도 0.7521)
파인튜닝 FINAL    : 0.9004  [95%CI 0.878~0.920]   (best epoch 3)
zero-shot 대비    : ΔFINAL=+0.0511 [+0.0259~+0.0741] | Δhard-FINAL=+0.0403 [+0.0171~+0.0615]
hard-FINAL        : 0.9100  (zero-shot 0.8697)  ← 비슷한 이미지끼리 비교한 더 어려운 지표
Recall@1          : 0.8560
NDCG@5            : 0.9448
캡션 교체 정확도  : 0.8718   (n=468)
학습 소요 시간    : 24.4분 (epoch별 평가 포함, 이어하기 시 누적)
